In [20]:
# import librarie
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import grangercausalitytests

In [21]:
# read data
data = pd.read_json("all_data.json")
data = data.sort_values(
    by=["Country Name", "year"],
)

In [22]:
# source: https://www.machinelearningplus.com/time-series/granger-causality-test-in-python/


def grangers_causation_matrix(
    data, variables, test="ssr_chi2test", verbose=False, maxlag=5
):
    """Check Granger Causality of all possible combinations of the Time series.
    The rows are the response variable, columns are predictors. The values in the table
    are the P-Values. P-Values lesser than the significance level (0.05), implies
    the Null Hypothesis that the coefficients of the corresponding past values is
    zero, that is, the X does not cause Y can be rejected.

    data      : pandas dataframe containing the time series variables
    variables : list containing names of the time series variables.
    """
    df = pd.DataFrame(
        np.zeros((len(variables), len(variables))), columns=variables, index=variables
    )
    for c in df.columns:
        for r in df.index:
            test_result = grangercausalitytests(
                data[[r, c]], maxlag=maxlag, verbose=False
            )
            p_values = [round(test_result[i + 1][0][test][1], 4) for i in range(maxlag)]
            if verbose:
                print(f"Y = {r}, X = {c}, P Values = {p_values}")
            min_p_value = np.min(p_values)
            df.loc[r, c] = min_p_value
    df.columns = [var + "_x" for var in variables]
    df.index = [var + "_y" for var in variables]
    return df

In [ ]:
features = [
    "GDP per Capita",
    "v4_prefixes_ris",
    "v6_prefixes_ris",
    "asns_ris",
    "v4_prefixes_stats",
    "v6_prefixes_stats",
    "asns_stats",
    "imputed speed",
    "imputed bandwidth",
]

# Ensure no missing data in selected features
test_data = data.dropna(subset=features)

# List of countries
countries = list(np.unique(test_data["Country Name"]))


# Granger causality matrix function
def grangers_causation_matrix(data, variables, maxlag=1, verbose=False):
    """
    Create a matrix of Granger causality test p-values.
    Each cell [Y, X] represents the p-value of the test
    whether X Granger-causes Y.
    """
    df = pd.DataFrame(
        np.zeros((len(variables), len(variables))), columns=variables, index=variables
    )

    for y in variables:
        for x in variables:
            if y != x:
                try:
                    test_result = grangercausalitytests(
                        data[[y, x]], maxlag=maxlag, verbose=verbose
                    )
                    p_value = test_result[maxlag][0][1]
                    df.loc[y, x] = p_value
                except:
                    df.loc[y, x] = np.nan
            else:
                df.loc[y, x] = np.nan

    return df


# Dictionary to store results per country
granger_results = {}

# Apply Granger causality per country
for country in countries:
    country_data = test_data[test_data["Country Name"] == country]

    # Ensure enough time points for analysis
    if len(country_data) > 5:
        try:
            causality_matrix = grangers_causation_matrix(
                country_data[features], features, maxlag=1, verbose=False
            )
            granger_results[country] = causality_matrix
        except Exception as e:
            print(f"Error processing {country}: {e}")

c:\Users\gebruikerr\Desktop\Lib\site-packages\statsmodels\tsa\stattools.py:1545: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
c:\Users\gebruikerr\Desktop\Lib\site-packages\statsmodels\tsa\stattools.py:1545: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
c:\Users\gebruikerr\Desktop\Lib\site-packages\statsmodels\tsa\stattools.py:1545: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
c:\Users\gebruikerr\Desktop\Lib\site-packages\statsmodels\tsa\stattools.py:1545: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
c:\Users\gebruikerr\Desktop\Lib\site-packages\statsmodels\tsa\stattools.py:1545: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
c:\Users\gebruikerr\Desktop\Lib\site-packages\statsmodels\tsa\stattools.py:1545: FutureWarning: verbose is deprecated si

In [ ]:
significant_results = pd.DataFrame(significant_results)

# Check if the DataFrame is not empty and has the expected columns
if not significant_results.empty and {"Cause x", "Effect y", "p-value"}.issubset(
    significant_results.columns
):
    # Group by cause-effect pairs and average the p-values
    summary_results = (
        significant_results[["Cause x", "Effect y", "p-value"]]
        .groupby(["Cause x", "Effect y"])
        .agg({"p-value": "mean"})
        .reset_index()
        .sort_values("p-value")
    )
    print(summary_results)
else:
    print(" No significant Granger causality results found.")

 No significant Granger causality results found.
